<a href="https://colab.research.google.com/github/swrobuts/dav/blob/main/notebooks/11_Fallstudie_Used_Cars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Fallstudie 3: Gebrauchtwagen-Preise

**Szenario**: Du arbeitest für AutoMarkt24, einen Gebrauchtwagenhändler. Ziel: Bereite den Datensatz für ein Preis-Vorhersagemodell vor.

**Vollständiger DAV-Workflow** – Bereinigen, Transformieren, Feature Engineering, ML-Vorbereitung.

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Echte Kursdaten + generierte Ergänzung
df_cars_xlsx = pd.read_excel(BASE_URL + "cars.xlsx")
df_extended = pd.read_csv(BASE_URL + "used_cars_extended.csv")

print('cars.xlsx (Originaldaten):')
print(df_cars_xlsx.shape)
print(df_cars_xlsx.head(3))
print()
print('used_cars_extended.csv (Generierte Daten):')
print(df_extended.shape)
print(df_extended.head(3))

## Phase 2: EDA

In [ ]:
df = df_extended.copy()

print('=== Daten-Überblick ===')
print(f'Shape: {df.shape}')
print()
print('Datentypen:')
print(df.dtypes)
print()
print('Fehlende Werte:')
print(df.isnull().sum()[df.isnull().sum()>0])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Preis-Verteilung
axes[0,0].hist(df['price'].clip(0, 50000), bins=40, color='#0389CF', edgecolor='white', alpha=0.8)
axes[0,0].set_title('Preis-Verteilung', fontweight='bold')
axes[0,0].set_xlabel('EUR')

# Laufleistung
df['mileage'].dropna().clip(0,250000).plot.hist(bins=40, ax=axes[0,1], color='#E8792F', alpha=0.8, edgecolor='white')
axes[0,1].set_title('Laufleistung', fontweight='bold')

# Baujahr
df['year'].value_counts().sort_index().plot.bar(ax=axes[0,2], color='#0389CF')
axes[0,2].set_title('Baujahre', fontweight='bold')
axes[0,2].set_xticklabels(axes[0,2].get_xticklabels(), rotation=45, ha='right', fontsize=7)

# Preis nach Marke (Top 8)
top_brands = df['brand'].value_counts().head(8).index
df[df['brand'].isin(top_brands)].boxplot(column='price', by='brand', ax=axes[1,0])
axes[1,0].set_title('Preis je Marke')
plt.sca(axes[1,0]); plt.title('Preis je Marke', fontweight='bold')
plt.gcf().suptitle('')
axes[1,0].set_xticklabels(axes[1,0].get_xticklabels(), rotation=45, ha='right')

# Kraftstofftyp
df['fuel_type'].value_counts().plot.bar(ax=axes[1,1], color='#0389CF')
axes[1,1].set_title('Kraftstofftypen', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=0)

# Scatter: Mileage vs Price
df_clean_scatter = df.dropna(subset=['mileage','price'])
axes[1,2].scatter(df_clean_scatter['mileage'], df_clean_scatter['price'],
                  alpha=0.3, color='#0389CF', s=10)
axes[1,2].set_title('Laufleistung vs. Preis', fontweight='bold')
axes[1,2].set_xlabel('km')
axes[1,2].set_ylabel('EUR')

plt.tight_layout()
plt.show()
print(f'\nKorrelation Mileage↔Preis: {df_clean_scatter["mileage"].corr(df_clean_scatter["price"]):.3f}')

## Phase 3: Data Preparation

In [ ]:
df_prep = df.copy()

# 1. Fehlende Werte
print(f'Fehlende vor Bereinigung: {df_prep.isnull().sum().sum()}')
df_prep['mileage'] = df_prep['mileage'].fillna(df_prep['mileage'].median())
print(f'Fehlende nach Bereinigung: {df_prep.isnull().sum().sum()}')

# 2. Ausreißer
Q1, Q99 = df_prep['price'].quantile([0.01, 0.99])
df_prep = df_prep[(df_prep['price'] >= Q1) & (df_prep['price'] <= Q99)]
df_prep = df_prep[df_prep['mileage'] >= 0]
print(f'Nach Ausreißer-Entfernung: {len(df_prep)} Zeilen')

# 3. Feature Engineering
df_prep['fahrzeugalter'] = 2026 - df_prep['year']
df_prep['preis_pro_km'] = (df_prep['price'] / df_prep['mileage'].replace(0, 1)).round(2)
df_prep['km_pro_jahr'] = (df_prep['mileage'] / df_prep['fahrzeugalter'].replace(0, 1)).round(0)
df_prep['is_electric'] = (df_prep['fuel_type'] == 'Elektro').astype(int)
df_prep['is_automatic'] = (df_prep['transmission'] == 'Automatik').astype(int)

print('\nNeue Features:')
print(df_prep[['brand','fahrzeugalter','preis_pro_km','km_pro_jahr','is_electric','is_automatic']].head(5))

In [ ]:
# 4. Encoding
cat_cols = ['brand', 'fuel_type', 'color']
le = LabelEncoder()
for col in cat_cols:
    df_prep[col+'_enc'] = le.fit_transform(df_prep[col].astype(str))
    print(f'{col}: {df_prep[col].nunique()} Kategorien kodiert')

# 5. Feature Scaling
num_features = ['mileage', 'fahrzeugalter', 'engine_size']
scaler = MinMaxScaler()
df_prep[num_features] = scaler.fit_transform(df_prep[num_features])
print('\nScaling abgeschlossen!')

# ML-Features auswählen
feature_cols = ['mileage','fahrzeugalter','engine_size','doors','is_electric','is_automatic',
                'brand_enc','fuel_type_enc']
target_col = 'price'

df_ml = df_prep[feature_cols + [target_col]].dropna()
print(f'\nML-bereiter Datensatz: {df_ml.shape}')

In [ ]:
# Train/Test Split
X = df_ml[feature_cols]
y = df_ml[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training-Set:   {X_train.shape[0]} Fahrzeuge')
print(f'Test-Set:       {X_test.shape[0]} Fahrzeuge')
print(f'\nTarget-Variable: price')
print(f'  Min: {y_train.min():.0f} EUR')
print(f'  Max: {y_train.max():.0f} EUR')
print(f'  Ø:   {y_train.mean():.0f} EUR')

# Korrelation mit Zielgröße
print('\nFeature-Korrelation mit Preis:')
corr_price = df_ml.corr()['price'].drop('price').abs().sort_values(ascending=False)
print(corr_price.round(3).to_string())

# Speichern
df_ml.to_csv("/tmp/used_cars_ml_ready.csv", index=False)
print('\n✅ ML-bereiter Datensatz gespeichert: /tmp/used_cars_ml_ready.csv')

## Erkenntnisse

1. **Fahrzeugalter** und **Laufleistung** korrelieren am stärksten negativ mit dem Preis
2. **Marke** (Brand) hat großen Einfluss – BMW/Mercedes deutlich teurer
3. **Elektrofahrzeuge** sind im Schnitt teurer als Verbrenner
4. Der Datensatz ist nun ML-bereit: numerisch, encodiert, skaliert, ohne Ausreißer

---
**Weiter mit:** `12_Wissenstests.ipynb`